# NeurIPS 2026 Smart Buildings — Submission Version 5

**Experiment:** Regret-aware HVAC allocation and slow online actuation calibration. This is a new control family relative to V2/V3 (per-AHU PID) and V4 (trend forecasts with supervisory hysteresis). V5 has NOT been scored on Kaggle. Lower is better.

## Architecture
- Direct per-zone sensor-band excursions, increasing convex tail-risk surrogate, and a whole-AHU cooling/heating allocation decision that preserves outlier demand.
- AHUs cool only; VAV reheat and the building's hot-water plant handle heating. Cold zones on mixed-demand AHUs receive restricted airflow rather than full cooling airflow.
- Fan pressure and hot-water reset allocated by 75th-percentile demand, active-zone coverage, and bounded worst-zone corrections; economizer considers exhaust fan power.
- Per-AHU strength [0.8, 1.4] adapts after 30 minutes of persistent unchanged mode, based on measured comfort-error reduction (no forecast, PID gains, or saved model).
- Uses each zone's current scoring setpoint sensors and a schedule fallback; true per-zone occupancy counts are not exposed, so no fabricated occupancy weights.

## Submission and tests
The self-contained notebook uses the official Kaggle inference-server interface: TransferPolicy.__init__, reset, get_action; produces float32 continuous actions in [-1,1] and int32 discrete actions in {0,1}. CPU-only, no internet, no external artifacts. Run the FINAL cell in Kaggle to start the inference server.

The flags ADAPT, RISK_WEIGHTING, ECONOMIZER support controlled ablations. The repository's test_version5.py is an offline synthetic contract test of the exact notebook policy code (variable shapes, metadata indexing, mixed hot/cold, physical actions, economizer, calibration, episode reset). It is not a 30-day public simulator evaluation. Run the official public scenarios before claiming improvement.

In [ ]:
"""V5: comfort-risk allocation with online actuator-strength calibration.

No PID, weather memorization, model weights, or look-ahead forecast.  Every
parameter is shared across buildings, and all device locations come from the
current episode's public metadata.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Allocate HVAC effort by per-zone comfort risk and observed response.

    The AHU only cools.  Every heating command is therefore implemented with
    VAV reheat and the shared hot-water system.  Unlike average-zone PID, an
    outlying zone cannot disappear in the mean; unlike V4, no temperature
    forecast is required.  A bounded six-step response check adjusts the
    actuation strength to buildings with different thermal envelopes.
    """

    ADAPT = True  # Ablation: set False to fix per-AHU strength at 1.0.
    RISK_WEIGHTING = True  # Ablation: False uses raw K excursions.
    ECONOMIZER = True

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.n_cont = int(m['n_continuous_action'])
        self.n_disc = int(m['n_discrete_action'])
        self.temp_actions = list(map(int, m['temp_indices']))
        self.press_actions = list(map(int, m['pressure_indices']))
        self.recirc_actions = list(map(int, m['recirculation_indices']))
        self.water_actions = list(map(int, m['water_indices']))
        self.n_ahu = len(self.temp_actions)
        self.water_temp = self.water_actions[0] if self.water_actions else None
        self.water_pressure = (self.press_actions[self.n_ahu]
                               if len(self.press_actions) > self.n_ahu else None)
        self.water_run = self.n_disc - 1
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        norm = m.get('observation_normalization_metadata', {}).get(
            'zone_air_temperature_sensor', {})
        self.zone_mean = float(norm.get('sample_mean', 0.0))
        self.zone_std = float(norm.get('sample_std', 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0.0:
            self.zone_std = 1.0

        def order(name):
            match = re.search(r'\d+', str(name))
            return (int(match.group()) if match else -1, str(name))

        assignments = m['ahu_zone_assignments']
        zone_action = m['zone_action_indices']
        self.ahus = []
        for i, ahu_id in enumerate(sorted(assignments, key=order)[:self.n_ahu]):
            zones = []
            for room in assignments[ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                temp = self.fields.get(vav + '_zone_air_temperature_sensor')
                if temp is None:
                    continue
                actions = zone_action.get(vav, {})
                zones.append({
                    'temp': temp,
                    'lo': self.fields.get(vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'hi': self.fields.get(vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper': actions.get('supply_air_damper_percentage_command'),
                    'valve': actions.get('reheat_valve_percentage_command'),
                })
            self.ahus.append({
                'zones': zones,
                'run': i,
                'supply': self.temp_actions[i],
                'pressure': self.press_actions[i] if i < len(self.press_actions) else None,
                'recirc': self.recirc_actions[i] if i < len(self.recirc_actions) else None,
                'outdoor': self.fields.get(f'ahs_{ahu_id}_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """The server may reuse a policy over episodes; never leak state."""
        self.strength = np.ones(len(self.ahus), dtype=np.float64)
        self.mode = ['off'] * len(self.ahus)
        self.mode_age = [0] * len(self.ahus)
        self.reference_severity = [0.0] * len(self.ahus)

    @staticmethod
    def _fraction(value):
        return float(np.clip(value, 0.0, 1.0))

    @classmethod
    def _command(cls, fraction):
        return 2.0 * cls._fraction(fraction) - 1.0

    def _schedule_band(self, obs):
        def phase(s, c, period):
            a, b = self.fields.get(s), self.fields.get(c)
            if a is None or b is None:
                return None
            x, y = float(obs[a]), float(obs[b])
            if not (np.isfinite(x) and np.isfinite(y)):
                return None
            return float(np.arctan2(x, y) % (2.0 * np.pi)) * period / (2.0 * np.pi)

        hour = phase('hod_sin_000', 'hod_cos_000', 24.0)
        day = phase('dow_sin_000', 'dow_cos_000', 7.0)
        occupied = (hour is not None and day is not None and
                    int(np.floor(day + 1e-6)) < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        temp = float(obs[zone['temp']]) * self.zone_std + self.zone_mean
        if not (np.isfinite(temp) and 240.0 <= temp <= 340.0):
            return None
        lo_idx, hi_idx = zone['lo'], zone['hi']
        lo = float(obs[lo_idx]) if lo_idx is not None else float('nan')
        hi = float(obs[hi_idx]) if hi_idx is not None else float('nan')
        if not (np.isfinite(lo) and np.isfinite(hi) and
                245.0 <= lo < hi <= 335.0 and hi - lo <= 35.0):
            lo, hi = fallback
        return {'zone': zone, 'temp': temp, 'lo': lo, 'hi': hi,
                'hot': max(0.0, temp - hi), 'cold': max(0.0, lo - temp)}

    def _risk(self, excursion):
        """A smooth surrogate of the scorer's unbounded tail comfort loss."""
        if not self.RISK_WEIGHTING:
            return excursion
        return excursion + max(0.0, excursion - 5.0) ** 2 / 5.0

    def _calibrate(self, i, mode, severity):
        """Bounded, six-step response calibration, not a PID integrator.

        Compare the current violation against its value 30 minutes earlier,
        *only* while the same mode persists. This is intentionally slow to
        avoid reacting to one noisy sensor reading or one cooling transition.
        """
        if mode == 'off' or mode != self.mode[i]:
            self.mode[i] = mode
            self.mode_age[i] = 0
            self.reference_severity[i] = severity
            return
        self.mode_age[i] += 1
        if self.mode_age[i] < 6:
            return
        improvement = self.reference_severity[i] - severity
        if self.ADAPT and severity >= 0.5:
            if improvement < 0.02:
                self.strength[i] = min(1.40, self.strength[i] * 1.04)
            elif improvement > 0.25:
                self.strength[i] = max(0.80, self.strength[i] * 0.97)
        self.reference_severity[i] = severity
        self.mode_age[i] = 0

    def _economize(self, outdoor, indoor, target, pressure_pa):
        # AHU recirculation is physically capped at 70%; outside air is NOT free
        # because the exhaust fan consumes electricity.
        if (not self.ECONOMIZER or not np.isfinite(outdoor) or
                outdoor >= indoor or indoor - outdoor <= pressure_pa * 0.00322):
            return 1.0
        useful_fraction = (indoor - max(target, 285.0)) / max(1e-6, indoor - outdoor)
        return self._command(1.0 - float(np.clip(useful_fraction, 0.30, 1.0)))

    def get_action(self, obs):
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous = np.full(self.n_cont, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_disc, dtype=np.int32)
        for idx in self.temp_actions:
            continuous[idx] = 1.0  # 305 K: AHUs cannot heat
        for idx in self.recirc_actions:
            continuous[idx] = 1.0  # max (physically capped) return-air fraction
        fallback = self._schedule_band(obs)
        water_demands = []

        for i, ahu in enumerate(self.ahus):
            zones = [self._read_zone(obs, z, fallback) for z in ahu['zones']]
            zones = [z for z in zones if z is not None]
            if not zones:
                self._calibrate(i, 'off', 0.0)
                continue
            hot = max(z['hot'] for z in zones)
            cold = max(z['cold'] for z in zones)
            if max(hot, cold) <= 0.12:
                self._calibrate(i, 'off', 0.0)
                continue

            hot_risk = sum(self._risk(z['hot']) for z in zones)
            cold_risk = sum(self._risk(z['cold']) for z in zones)
            # Protect an extreme tail; use total demand when excursions are
            # similar. The AHU cannot heat, so cooling is a costly global choice.
            cool = hot > 0.12 and (hot_risk + 0.7 * self._risk(hot) >=
                                   0.65 * cold_risk or hot >= 2.5)
            mode = 'cool' if cool else 'heat'
            severity = hot if cool else cold
            self._calibrate(i, mode, severity)
            gain = float(self.strength[i])
            discrete[ahu['run']] = 1

            # Fan power is expensive. Run near the 200 Pa floor until there is
            # meaningful multi-zone demand or an extreme comfort violation.
            demands = [z['hot'] if cool else z['cold'] for z in zones]
            significant = sum(x > 0.3 for x in demands) / len(demands)
            representative = float(np.percentile(demands, 75))
            pressure = 200.0 + 1450.0 * self._fraction(
                gain * (representative + 0.35 * severity) / 3.2) * (
                    0.40 + 0.60 * np.sqrt(significant))
            pressure = float(np.clip(pressure, 200.0, 1800.0))
            if ahu['pressure'] is not None:
                continuous[ahu['pressure']] = float(np.clip(pressure / 1000.0 - 1.0,
                                                           -1.0, 1.0))

            if cool:
                # Direct cooling-only supply reset, not the PID output and not
                # a forecast. 285..305 K is the starter-kit native range.
                target = float(np.clip(294.0 - 3.0 * gain *
                                       (0.65 * representative + 0.35 * hot),
                                       285.0, 296.0))
                continuous[ahu['supply']] = float(np.clip((target - 285.0) / 10.0 - 1.0,
                                                           -1.0, 1.0))
                outdoor_idx = ahu['outdoor']
                outside = float(obs[outdoor_idx]) if outdoor_idx is not None else np.nan
                inside = float(np.mean([z['temp'] for z in zones]))
                if ahu['recirc'] is not None:
                    continuous[ahu['recirc']] = self._economize(outside, inside,
                                                                target, pressure)

            for z in zones:
                zone = z['zone']
                # Avoid deliberately cooling a cold zone while another zone
                # needs cooling: a cold VAV uses only the minimum airflow
                # needed to transport reheat, not a full-open damper.
                cooling_fraction = self._fraction(gain * (z['hot'] + 0.05) / 2.4) if cool else 0.0
                heating_fraction = self._fraction(gain * (z['cold'] + 0.05) / 2.6)
                airflow_fraction = max(cooling_fraction,
                                       0.18 * heating_fraction if cool else heating_fraction)
                if zone['damper'] is not None:
                    continuous[zone['damper']] = self._command(airflow_fraction)
                if z['cold'] > 0.12 and zone['valve'] is not None:
                    valve = self._fraction(gain * (z['cold'] + 0.10) / 2.2)
                    continuous[zone['valve']] = self._command(valve)
                    water_demands.append((z['cold'], valve))

        if water_demands and 0 <= self.water_run < self.n_disc:
            discrete[self.water_run] = 1
            shortfalls = [x for x, _ in water_demands]
            valve_load = [v for _, v in water_demands]
            p75 = float(np.percentile(shortfalls, 75))
            # Robust to one unreachable cold zone, without neglecting a
            # building-wide shortage. The full water range is 310..350 K.
            demand = self._fraction(p75 / 2.8 + 0.15 * max(shortfalls) / 4.0)
            target_water = 320.0 + 28.0 * demand
            if self.water_temp is not None:
                continuous[self.water_temp] = self._command((target_water - 310.0) / 40.0)
            if self.water_pressure is not None:
                continuous[self.water_pressure] = self._command(
                    max(0.38, 0.35 + 0.65 * float(np.mean(valve_load))))

        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
# Kaggle executes the public smoke rollout and hidden evaluation.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()